
# CSCI 408 — BLIP: Raw vs Current vs Fine-Tuned

## Goal

Compare **three versions of the same BLIP report-generation architecture** on the same 30 unseen Indiana Chest X-ray examples:

1. **Raw / randomized BLIP**  
   Same architecture/configuration as the current model, but initialized with random weights, then trained on the 270-example training split.

2. **Current model (baseline)**  
   `nathansutton/generate-cxr` exactly as used in the current project, with **no extra training on our 270 examples**.

3. **Fine-tuned current model**  
   Start from `nathansutton/generate-cxr`, then fine-tune it on the **same 270-example training split** used for the raw model.

### Dataset protocol

- 3 classes: `Normal`, `Cardiomegaly`, `Pleural Effusion`
- 100 examples per class = **300 total**
- **270 training** examples = 90 per class
- **30 testing** examples = 10 per class
- The same fixed test set is used for all three models.
- Test images are never used for training.

### Important interpretation note

This notebook compares generated text using **automatic text-similarity metrics** and side-by-side outputs.  
It is **not a clinical validation** and does not claim that a model is medically correct. We are not radiologists; the final “best model” statement is only an approximate comparison against the reference reports.


In [ ]:

# Run once in a fresh Colab / Jupyter environment.
# transformers<5 keeps compatibility close to the original project notebook.
!pip -q install "transformers>=4.45,<5" rouge-score bert-score nltk accelerate sentencepiece



## 1. Imports and experiment configuration

The generation settings intentionally match the current project as closely as possible:
`max_new_tokens=120`, `num_beams=3`, `repetition_penalty=2.0`.


In [1]:

import os
import gc
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import (
    BlipProcessor,
    BlipConfig,
    BlipForConditionalGeneration,
)
from transformers.optimization import Adafactor

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------
# Paths
# ----------------------------
PROJECTIONS_CSV = "indiana_projections.csv"
REPORTS_CSV = "indiana_reports.csv"
IMAGE_DIR = Path("images")

# ----------------------------
# Model
# ----------------------------
MODEL_ID = "nathansutton/generate-cxr"

# ----------------------------
# Split
# ----------------------------
N_PER_CLASS = 100
TRAIN_SIZE = 270
TEST_SIZE = 30

# ----------------------------
# Training
# ----------------------------
# BLIP is large. Batch size 1 + gradient accumulation is safer on common GPUs.
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

# You can increase epochs later if your GPU/time allows.
RAW_EPOCHS = 3
FINETUNE_EPOCHS = 3

RAW_LR = 1e-4
FINETUNE_LR = 1e-5
MAX_TEXT_LENGTH = 160

# ----------------------------
# Generation
# ----------------------------
MAX_NEW_TOKENS = 120
NUM_BEAMS = 3
REPETITION_PENALTY = 2.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type != "cuda":
    print(
        "WARNING: Training a ~0.5B parameter BLIP model on CPU is impractically slow. "
        "Use a CUDA GPU runtime if possible."
    )


c:\Users\danik\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu



## 2. Build the exact balanced 300-example dataset

This follows the target-class logic from the current project, with one important safeguard for report generation:

- create the text target from `findings`, falling back to `impression`;
- remove rows where the report text is still missing/empty;
- then sample 100 usable examples from each target class.

This guarantees that every one of the 300 selected images has a target report.


In [2]:

proj_df = pd.read_csv(PROJECTIONS_CSV)
reports_df = pd.read_csv(REPORTS_CSV)

df = pd.merge(proj_df, reports_df, on="uid")
df_frontal = df[df["projection"] == "Frontal"].copy()

def get_single_label(tags):
    tags = str(tags).lower()

    has_cardiomegaly = "cardiomegaly" in tags
    has_effusion = ("pleural effusion" in tags) or ("effusion" in tags)
    has_normal = "normal" in tags

    # Keep only one of the three target labels.
    if sum([has_cardiomegaly, has_effusion, has_normal]) > 1:
        return "Exclude"
    elif has_cardiomegaly:
        return "Cardiomegaly"
    elif has_effusion:
        return "Pleural Effusion"
    elif has_normal:
        return "Normal"
    else:
        return "Exclude"

df_frontal["y_label"] = df_frontal["MeSH"].apply(get_single_label)
df_filtered = df_frontal[df_frontal["y_label"] != "Exclude"].copy()

# Same report target idea as the current project:
# findings first, impression as fallback.
df_filtered["y_report"] = df_filtered["findings"].fillna(df_filtered["impression"])
df_filtered["y_report"] = df_filtered["y_report"].astype("string").str.strip()

# A report-generation training example must actually have a report.
df_filtered = df_filtered[
    df_filtered["y_report"].notna()
    & (df_filtered["y_report"] != "")
    & (df_filtered["y_report"].str.lower() != "nan")
].copy()

available = df_filtered["y_label"].value_counts()
print("Usable examples available before balanced sampling:")
display(available.to_frame("count"))

for label in ["Normal", "Cardiomegaly", "Pleural Effusion"]:
    assert available[label] >= N_PER_CLASS, f"Not enough usable rows for {label}"

# Exactly 100 usable examples per class.
df_300 = (
    df_filtered
    .groupby("y_label", group_keys=False)
    .sample(n=N_PER_CLASS, random_state=SEED)
    [["filename", "y_label", "y_report"]]
    .rename(columns={"filename": "image"})
    .reset_index(drop=True)
)

assert len(df_300) == 300
assert (df_300["y_label"].value_counts() == 100).all()
assert df_300["y_report"].notna().all()

print("\nBalanced selected dataset:")
display(df_300["y_label"].value_counts().sort_index().to_frame("count"))


Usable examples available before balanced sampling:


,count
y_label,
Normal,1366
Cardiomegaly,284
Pleural Effusion,101



Balanced selected dataset:


,count
y_label,
Cardiomegaly,100
Normal,100
Pleural Effusion,100



## 3. Fixed 270/30 split

Because the full set contains exactly 100 examples per class, a stratified 10% test split produces exactly:

- 90 train + 10 test `Normal`
- 90 train + 10 test `Cardiomegaly`
- 90 train + 10 test `Pleural Effusion`

The split CSVs are saved so every later run uses the same samples.


In [3]:

train_df, test_df = train_test_split(
    df_300,
    test_size=TEST_SIZE,
    stratify=df_300["y_label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert len(train_df) == TRAIN_SIZE
assert len(test_df) == TEST_SIZE

# With 100/class and a 10% stratified split this should be exactly 90/10 per class.
assert (train_df["y_label"].value_counts() == 90).all()
assert (test_df["y_label"].value_counts() == 10).all()

train_df.to_csv("report_train_270.csv", index=False)
test_df.to_csv("report_test_30.csv", index=False)

print(f"Train = {len(train_df)} | Test = {len(test_df)}")
print("\nTrain distribution:")
display(train_df["y_label"].value_counts().sort_index().to_frame("count"))
print("\nTest distribution:")
display(test_df["y_label"].value_counts().sort_index().to_frame("count"))


Train = 270 | Test = 30

Train distribution:


,count
y_label,
Cardiomegaly,90
Normal,90
Pleural Effusion,90



Test distribution:


,count
y_label,
Cardiomegaly,10
Normal,10
Pleural Effusion,10



## 4. Verify the image files

This notebook assumes the same layout as the current project:

```text
project/
├── indiana_projections.csv
├── indiana_reports.csv
├── images/
│   ├── 1_IM-0001-4001.dcm.png
│   └── ...
└── this_notebook.ipynb
```

The experiment stops immediately if any selected image is missing; silently dropping images would break the exact 270/30 protocol.


In [4]:

all_required = pd.concat([train_df, test_df], ignore_index=True)
missing_images = [
    name for name in all_required["image"]
    if not (IMAGE_DIR / name).exists()
]

print(f"Required images: {len(all_required)}")
print(f"Missing images:  {len(missing_images)}")

if missing_images:
    print("First missing filenames:")
    print("\n".join(missing_images[:20]))
    raise FileNotFoundError(
        f"{len(missing_images)} required images are missing from {IMAGE_DIR.resolve()}. "
        "Put the Indiana X-ray PNG files into the images/ folder before continuing."
    )

print("All 300 required image files were found.")


Required images: 300
Missing images:  300
First missing filenames:
967_IM-2457-2002.dcm.png
3414_IM-1650-1001.dcm.png
1414_IM-0264-1001.dcm.png
3915_IM-1990-1001.dcm.png
1053_IM-0040-1001.dcm.png
2308_IM-0883-1001.dcm.png
25_IM-1024-2001.dcm.png
3690_IM-1841-1001.dcm.png
2217_IM-0822-0001-0002.dcm.png
2220_IM-0824-4004.dcm.png
2848_IM-1256-1001.dcm.png
1698_IM-0458-1001.dcm.png
3234_IM-1531-1001.dcm.png
1482_IM-0313-1001.dcm.png
797_IM-2332-1001.dcm.png
1743_IM-0489-4004.dcm.png
188_IM-0569-1001.dcm.png
1278_IM-0185-1001.dcm.png
1038_IM-0029-1001.dcm.png
275_IM-1200-1001.dcm.png


FileNotFoundError: 300 required images are missing from C:\Users\danik\Desktop\Проекты\Medical Image\images. Put the Indiana X-ray PNG files into the images/ folder before continuing.


## 5. Processor, Dataset, and DataLoader

Training uses teacher forcing: the image is the visual input and the reference report supplies decoder tokens/labels.

All three variants use the **same processor/tokenizer** and generation settings.


In [6]:

processor = BlipProcessor.from_pretrained(MODEL_ID)

class CXRReportDataset(Dataset):
    def __init__(self, dataframe, image_dir):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "image_path": str(self.image_dir / row["image"]),
            "report": str(row["y_report"]),
            "label": row["y_label"],
            "filename": row["image"],
        }

def collate_train(batch):
    images = [
        Image.open(item["image_path"]).convert("RGB")
        for item in batch
    ]
    reports = [item["report"] for item in batch]

    enc = processor(
        images=images,
        text=reports,
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="pt",
    )

    labels = enc["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    labels[labels == pad_id] = -100
    enc["labels"] = labels
    return enc

train_dataset = CXRReportDataset(train_df, IMAGE_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_train,
)

print("Training batches:", len(train_loader))


Training batches: 270



## 6. Training and generation helper functions

Memory choices:

- models are handled **sequentially**, not all kept on the GPU together;
- Adafactor is used because it is much lighter on optimizer-state memory than standard AdamW for a large model;
- CUDA mixed precision is enabled when available;
- gradient checkpointing is attempted when supported.

No test example is passed to the training function.


In [7]:

def cleanup_model(model=None):
    if model is not None:
        try:
            model.to("cpu")
        except Exception:
            pass
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def prepare_for_training(model):
    model = model.to(device)

    # Reduce activation memory if supported by this transformers version/model.
    try:
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing: enabled")
    except Exception as e:
        print("Gradient checkpointing not enabled:", type(e).__name__)

    # Caching is unnecessary during training and can conflict with checkpointing.
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    return model


def train_blip(model, train_loader, epochs, lr, run_name):
    model = prepare_for_training(model)

    optimizer = Adafactor(
        model.parameters(),
        lr=lr,
        scale_parameter=False,
        relative_step=False,
        warmup_init=False,
        weight_decay=0.0,
    )

    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen_steps = 0

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(
                    pixel_values=batch["pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch.get("attention_mask"),
                    labels=batch["labels"],
                )
                loss = outputs.loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            should_step = (
                step % GRAD_ACCUM_STEPS == 0
                or step == len(train_loader)
            )

            if should_step:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * GRAD_ACCUM_STEPS
            seen_steps += 1

            if step % 25 == 0 or step == len(train_loader):
                print(
                    f"{run_name} | epoch {epoch}/{epochs} | "
                    f"step {step}/{len(train_loader)} | "
                    f"mean loss {running_loss / seen_steps:.4f}"
                )

        epoch_loss = running_loss / max(seen_steps, 1)
        history.append({"epoch": epoch, "loss": epoch_loss})
        print(f"{run_name} | epoch {epoch} complete | loss={epoch_loss:.4f}")

    return model, pd.DataFrame(history)


@torch.inference_mode()
def generate_report(model, image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    generated_ids = model.generate(
        pixel_values=pixel_values,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        repetition_penalty=REPETITION_PENALTY,
    )

    return processor.decode(
        generated_ids[0],
        skip_special_tokens=True
    ).strip()


def generate_for_test_set(model, model_name):
    outputs = []

    print(f"\nGenerating 30 reports with: {model_name}")
    for i, row in test_df.iterrows():
        text = generate_report(model, IMAGE_DIR / row["image"])
        outputs.append(text)

        print(
            f"[{i+1:02d}/{len(test_df)}] "
            f"{row['y_label']} | {row['image']}"
        )

    return outputs



# MODEL 2 — Current model / baseline

We evaluate the project's current `nathansutton/generate-cxr` checkpoint **before any training on our 270 examples**.

This gives the baseline output.


In [ ]:

cleanup_model()

baseline_model = BlipForConditionalGeneration.from_pretrained(MODEL_ID)
baseline_model = baseline_model.to(device)

baseline_outputs = generate_for_test_set(
    baseline_model,
    "Current model (baseline)"
)

# Do not keep the large model in GPU memory.
cleanup_model(baseline_model)
baseline_model = None

print("\nBaseline generation complete.")



# MODEL 1 — Raw/randomized weights + training on 270

`BlipForConditionalGeneration(config)` constructs the **same BLIP architecture/configuration without loading the checkpoint weights**.

That means the weights start randomized. It is then trained only on our 270 training examples.

This is intentionally a difficult condition: 270 examples are extremely small for learning a ~0.5B-parameter vision-language model from scratch. Poor performance is therefore meaningful evidence about the value of pretraining, not a bug.


In [ ]:

cleanup_model()

raw_config = BlipConfig.from_pretrained(MODEL_ID)
raw_model = BlipForConditionalGeneration(raw_config)

print("Raw model created from configuration only.")
print("No pretrained checkpoint weights were loaded.")

raw_model, raw_history = train_blip(
    raw_model,
    train_loader,
    epochs=RAW_EPOCHS,
    lr=RAW_LR,
    run_name="RAW"
)

display(raw_history)

raw_outputs = generate_for_test_set(
    raw_model,
    "Raw randomized model after training on 270"
)

# Optional: save locally if you want to reuse it later.
RAW_SAVE_DIR = "raw_blip_270_checkpoint"
raw_model.save_pretrained(RAW_SAVE_DIR)
processor.save_pretrained(RAW_SAVE_DIR)
print("Saved:", RAW_SAVE_DIR)

cleanup_model(raw_model)
raw_model = None



# MODEL 3 — Fine-tuned current model + training on the same 270

This model starts from the exact current checkpoint and receives extra training on the 270-example Indiana training set.

The raw and fine-tuned models therefore see the **same 270 examples**, while the baseline sees none of them.


In [ ]:

cleanup_model()

finetuned_model = BlipForConditionalGeneration.from_pretrained(MODEL_ID)

finetuned_model, finetune_history = train_blip(
    finetuned_model,
    train_loader,
    epochs=FINETUNE_EPOCHS,
    lr=FINETUNE_LR,
    run_name="FINE-TUNE"
)

display(finetune_history)

finetuned_outputs = generate_for_test_set(
    finetuned_model,
    "Fine-tuned current model"
)

FT_SAVE_DIR = "finetuned_blip_270_checkpoint"
finetuned_model.save_pretrained(FT_SAVE_DIR)
processor.save_pretrained(FT_SAVE_DIR)
print("Saved:", FT_SAVE_DIR)

cleanup_model(finetuned_model)
finetuned_model = None



## 7. Put all 30 outputs side by side

Every row below corresponds to one **unseen** test X-ray.

The table contains:

- ground-truth class label;
- reference report;
- raw/randomized model report;
- current baseline report;
- fine-tuned report.


In [ ]:

comparison_df = test_df.copy()

comparison_df["Raw_Randomized"] = raw_outputs
comparison_df["Current_Baseline"] = baseline_outputs
comparison_df["Fine_Tuned"] = finetuned_outputs

# Helpful safety check: all three models must have exactly one output per test image.
assert len(comparison_df) == 30
assert comparison_df["Raw_Randomized"].notna().all()
assert comparison_df["Current_Baseline"].notna().all()
assert comparison_df["Fine_Tuned"].notna().all()

pd.set_option("display.max_colwidth", 220)

display(
    comparison_df[
        [
            "image",
            "y_label",
            "y_report",
            "Raw_Randomized",
            "Current_Baseline",
            "Fine_Tuned",
        ]
    ]
)

comparison_df.to_csv("comparison_all_30_outputs.csv", index=False)
print("Saved: comparison_all_30_outputs.csv")



## 8. Automatic approximate comparison

We use three non-clinical text metrics:

- **BLEU-4** — exact/local n-gram overlap with the reference report;
- **ROUGE-L F1** — longest-sequence overlap;
- **BERTScore F1** — general semantic similarity.

Higher is better for each metric.

### Why this is only approximate

A generated radiology report can use different wording and still be medically equivalent, while a report can also obtain decent text overlap and still contain a clinically important error.

Therefore these metrics compare **similarity to the dataset report**, not diagnostic safety or medical correctness.


In [ ]:

smoother = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def sentence_bleu4(reference, candidate):
    ref_tokens = str(reference).lower().split()
    cand_tokens = str(candidate).lower().split()

    if not cand_tokens:
        return 0.0

    return sentence_bleu(
        [ref_tokens],
        cand_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smoother,
    )

def sentence_rouge_l(reference, candidate):
    return rouge.score(
        str(reference),
        str(candidate)
    )["rougeL"].fmeasure

model_columns = {
    "Raw randomized + 270 train": "Raw_Randomized",
    "Current baseline": "Current_Baseline",
    "Fine-tuned + 270 train": "Fine_Tuned",
}

references = comparison_df["y_report"].astype(str).tolist()

for readable_name, col in model_columns.items():
    comparison_df[f"{col}_BLEU4"] = [
        sentence_bleu4(ref, pred)
        for ref, pred in zip(references, comparison_df[col].astype(str))
    ]

    comparison_df[f"{col}_ROUGE_L"] = [
        sentence_rouge_l(ref, pred)
        for ref, pred in zip(references, comparison_df[col].astype(str))
    ]

    # BERTScore is computed over all 30 pairs together.
    _, _, f1 = bert_score(
        comparison_df[col].astype(str).tolist(),
        references,
        model_type="distilbert-base-uncased",
        lang="en",
        verbose=False,
        device=str(device),
    )

    comparison_df[f"{col}_BERTScore_F1"] = f1.cpu().numpy()

print("Per-example metrics calculated.")


In [ ]:

summary_rows = []

for readable_name, col in model_columns.items():
    summary_rows.append({
        "Model": readable_name,
        "BLEU-4": comparison_df[f"{col}_BLEU4"].mean(),
        "ROUGE-L F1": comparison_df[f"{col}_ROUGE_L"].mean(),
        "BERTScore F1": comparison_df[f"{col}_BERTScore_F1"].mean(),
    })

metrics_summary = pd.DataFrame(summary_rows)

# Rank each metric separately (1 = best).
for metric in ["BLEU-4", "ROUGE-L F1", "BERTScore F1"]:
    metrics_summary[f"{metric} rank"] = metrics_summary[metric].rank(
        ascending=False,
        method="min"
    )

metrics_summary["Mean metric rank"] = metrics_summary[
    ["BLEU-4 rank", "ROUGE-L F1 rank", "BERTScore F1 rank"]
].mean(axis=1)

metrics_summary = metrics_summary.sort_values(
    ["Mean metric rank", "BERTScore F1"],
    ascending=[True, False]
).reset_index(drop=True)

display(metrics_summary)

metrics_summary.to_csv("metrics_summary.csv", index=False)
comparison_df.to_csv("comparison_all_30_outputs_with_metrics.csv", index=False)

print("Saved: metrics_summary.csv")
print("Saved: comparison_all_30_outputs_with_metrics.csv")



## 9. Visual comparison of the three models


In [ ]:

plot_df = metrics_summary.set_index("Model")[
    ["BLEU-4", "ROUGE-L F1", "BERTScore F1"]
]

ax = plot_df.plot(kind="bar", figsize=(11, 5))
ax.set_title("Raw vs Current vs Fine-Tuned — 30 unseen X-rays")
ax.set_ylabel("Score (higher is better)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
ax.legend(loc="best")
plt.tight_layout()
plt.show()



## 10. Approximate winner statement

This cell deliberately uses **metric ranks**, rather than averaging BLEU, ROUGE and BERTScore raw values directly (their numerical scales are not equivalent).

The result is phrased as a **text-similarity result**, not a medical judgment.


In [ ]:

best_row = metrics_summary.iloc[0]

print("=" * 90)
print("APPROXIMATE NON-CLINICAL RESULT")
print("=" * 90)
print(f"Best overall by mean rank across the 3 text-similarity metrics:")
print(f"  {best_row['Model']}")
print()
print(
    "Interpretation: on these 30 held-out examples, this model's generated reports "
    "are approximately the closest to the dataset reference reports according to "
    "BLEU-4, ROUGE-L and BERTScore."
)
print()
print(
    "This does NOT prove that the model is medically safest or diagnostically correct. "
    "A radiologist/clinical expert would be required for clinical evaluation."
)
print("=" * 90)



## 11. Inspect the actual outputs sample by sample

Use this section for the presentation/report. It lets us say things such as:

- whether the raw model produces broken/repetitive text;
- whether the current model produces coherent radiology-style language;
- whether fine-tuning makes outputs closer to the Indiana reference wording;
- where one model adds unsupported details or misses information.

Again, these are **qualitative observations**, not a medical diagnosis.


In [ ]:

for i, row in comparison_df.iterrows():
    print("=" * 110)
    print(f"TEST SAMPLE {i+1:02d}/30")
    print(f"Image: {row['image']}")
    print(f"Dataset label: {row['y_label']}")
    print("-" * 110)

    print("\nREFERENCE REPORT:")
    print(row["y_report"])

    print("\nRAW / RANDOMIZED + TRAINED ON 270:")
    print(row["Raw_Randomized"])

    print("\nCURRENT BASELINE:")
    print(row["Current_Baseline"])

    print("\nFINE-TUNED + TRAINED ON 270:")
    print(row["Fine_Tuned"])

    print()



## 12. Optional human review sheet

Because we are not doctors, do **not** score “medical correctness”.

If the team wants a simple manual comparison, reviewers can score only surface-level properties such as:

- readability/coherence;
- obvious repetition;
- similarity to the supplied reference report;
- whether the output is empty/broken.

The blank columns below can be filled manually.


In [ ]:

human_review = comparison_df[
    [
        "image",
        "y_label",
        "y_report",
        "Raw_Randomized",
        "Current_Baseline",
        "Fine_Tuned",
    ]
].copy()

human_review["Raw_coherence_1_to_5"] = ""
human_review["Baseline_coherence_1_to_5"] = ""
human_review["FineTuned_coherence_1_to_5"] = ""
human_review["Reviewer_notes"] = ""

human_review.to_csv("human_review_30_template.csv", index=False)

display(human_review.head())
print("Saved: human_review_30_template.csv")



# What to report in the project

A concise description for the experiment:

> We compared three variants of the same BLIP-based chest X-ray report generator on a fixed balanced subset of 300 Indiana Chest X-ray cases. A randomized-weight BLIP model and a fine-tuned pretrained BLIP model were trained on the same 270-image training split, while the existing pretrained model was kept unchanged as a baseline. All three models were evaluated on the same 30 held-out images (10 per target class). We compared the generated reports against the reference reports using BLEU-4, ROUGE-L, and BERTScore, and inspected outputs qualitatively. The comparison is non-clinical and is intended to study the effect of pretraining and dataset-specific fine-tuning rather than establish diagnostic accuracy.

### Expected scientific question

**Does pretraining help compared with learning from only 270 images, and does additional fine-tuning on the project dataset improve report similarity on held-out Indiana cases?**

### Important limitation

The current checkpoint was pretrained/fine-tuned on external chest X-ray report data, while the randomized model receives only 270 local training examples. Therefore this is primarily an experiment about the value of **pretraining + domain adaptation**, not a claim that the models received equal amounts of prior information.
